# Анализ распределения dataset2_split

Проверяем распределение по сплитам: кол-во пациентов, последовательностей и кадров.

In [1]:
import os
import json
import re
from collections import defaultdict
from pathlib import Path
import pandas as pd

BASE = Path('/home/dsa/stenosis/data/dataset2_split')
SPLITS = {'train': 'train/images', 'valid': 'valid/images', 'test': 'test/images'}

In [2]:
# Парсим имя файла: 14_003_1_0001_bmp_jpg.rf.xxx.jpg
# patient = 14_003, sequence = 1, frame = 0001

def parse_filename(fname):
    """Возвращает (patient_id, sequence_id, frame_id) из имени файла."""
    # паттерн: {prefix}_{patnum}_{seq}_{frame}_bmp_jpg.rf.{hash}.jpg
    m = re.match(r'^(\d+_\d+)_(\d+)_(\d+)_bmp_jpg', fname)
    if m:
        return m.group(1), int(m.group(2)), int(m.group(3))
    return None, None, None

# Собираем статистику по каждому сплиту
split_data = {}
for split_name, img_dir in SPLITS.items():
    img_path = BASE / img_dir
    files = sorted(os.listdir(img_path))
    
    patients = set()
    sequences = set()  # (patient, seq)
    frames = 0
    
    for f in files:
        patient, seq, frame = parse_filename(f)
        if patient:
            patients.add(patient)
            sequences.add((patient, seq))
            frames += 1
    
    split_data[split_name] = {
        'patients': patients,
        'sequences': sequences,
        'n_patients': len(patients),
        'n_sequences': len(sequences),
        'n_frames': frames,
    }

print('Готово: данные собраны')

Готово: данные собраны


In [3]:
# Сводная таблица по сплитам
summary = pd.DataFrame([
    {
        'Split': split_name,
        'Пациентов': split_data[split_name]['n_patients'],
        'Последовательностей': split_data[split_name]['n_sequences'],
        'Кадров': split_data[split_name]['n_frames'],
    }
    for split_name in ['train', 'valid', 'test']
])

# Добавляем строку Total
total = pd.DataFrame([{
    'Split': 'TOTAL',
    'Пациентов': summary['Пациентов'].sum(),
    'Последовательностей': summary['Последовательностей'].sum(),
    'Кадров': summary['Кадров'].sum(),
}])
summary = pd.concat([summary, total], ignore_index=True)
summary

,Split,Пациентов,Последовательностей,Кадров
0,train,44,151,5817
1,valid,10,32,1254
2,test,10,31,1252
3,TOTAL,64,214,8323


In [4]:
# Процентное распределение кадров по сплитам
total_frames = sum(split_data[s]['n_frames'] for s in SPLITS)
for s in ['train', 'valid', 'test']:
    d = split_data[s]
    pct = d['n_frames'] / total_frames * 100
    print(f"{s:6s}: {d['n_frames']:5d} кадров ({pct:.1f}%), "
          f"{d['n_patients']:2d} пациентов, {d['n_sequences']:3d} последовательностей")

train :  5817 кадров (69.9%), 44 пациентов, 151 последовательностей
valid :  1254 кадров (15.1%), 10 пациентов,  32 последовательностей
test  :  1252 кадров (15.0%), 10 пациентов,  31 последовательностей


In [5]:
# Детальная таблица: по каждому пациенту — кол-во последовательностей и кадров
rows = []
for split_name in ['train', 'valid', 'test']:
    img_path = BASE / SPLITS[split_name]
    files = sorted(os.listdir(img_path))
    
    patient_seqs = defaultdict(set)
    patient_frames = defaultdict(int)
    
    for f in files:
        patient, seq, frame = parse_filename(f)
        if patient:
            patient_seqs[patient].add(seq)
            patient_frames[patient] += 1
    
    for patient in sorted(patient_seqs.keys()):
        rows.append({
            'Split': split_name,
            'Patient': patient,
            'Sequences': len(patient_seqs[patient]),
            'Frames': patient_frames[patient],
        })

df_detail = pd.DataFrame(rows)
df_detail

,Split,Patient,Sequences,Frames
0,train,14_003,2,44
1,train,14_007,5,374
2,train,14_010,2,82
3,train,14_014,1,23
4,train,14_016,1,70
...,...,...,...,...
59,test,14_066,6,196
60,test,14_083,3,84
61,test,14_091,1,11
62,test,14_096,5,235


In [6]:
# Статистика по кадрам на пациента в каждом сплите
print('Кадров на пациента (min / mean / max):')
for split_name in ['train', 'valid', 'test']:
    sub = df_detail[df_detail['Split'] == split_name]['Frames']
    print(f"  {split_name:6s}: {sub.min():4d} / {sub.mean():.1f} / {sub.max():4d}")

print()
print('Последовательностей на пациента (min / mean / max):')
for split_name in ['train', 'valid', 'test']:
    sub = df_detail[df_detail['Split'] == split_name]['Sequences']
    print(f"  {split_name:6s}: {sub.min():4d} / {sub.mean():.1f} / {sub.max():4d}")

Кадров на пациента (min / mean / max):
  train :   23 / 132.2 /  543
  valid :   14 / 125.4 /  332
  test  :   11 / 125.2 /  326

Последовательностей на пациента (min / mean / max):
  train :    1 / 3.4 /   15
  valid :    1 / 3.2 /    7
  test  :    1 / 3.1 /    6


In [7]:
# Проверяем, что нет пересечения пациентов между сплитами
train_p = split_data['train']['patients']
valid_p = split_data['valid']['patients']
test_p = split_data['test']['patients']

print('Пересечение train & valid:', train_p & valid_p)
print('Пересечение train & test: ', train_p & test_p)
print('Пересечение valid & test: ', valid_p & test_p)

if not (train_p & valid_p) and not (train_p & test_p) and not (valid_p & test_p):
    print('\n✅ Утечки пациентов между сплитами НЕТ')
else:
    print('\n❌ Обнаружена утечка пациентов!')

Пересечение train & valid: set()
Пересечение train & test:  set()
Пересечение valid & test:  set()

✅ Утечки пациентов между сплитами НЕТ


In [8]:
# Сверяем с manifest
with open(BASE / 'patient_split_manifest.json') as f:
    manifest = json.load(f)

for split_name in ['train', 'valid', 'test']:
    manifest_patients = set(manifest['splits'][split_name]['patients'])
    actual_patients = split_data[split_name]['patients']
    manifest_count = manifest['splits'][split_name]['image_count']
    actual_count = split_data[split_name]['n_frames']
    
    match_p = '✅' if manifest_patients == actual_patients else '❌'
    match_c = '✅' if manifest_count == actual_count else '❌'
    
    print(f"{split_name}: пациенты {match_p} (manifest={len(manifest_patients)}, actual={len(actual_patients)}), "
          f"кадры {match_c} (manifest={manifest_count}, actual={actual_count})")

train: пациенты ✅ (manifest=44, actual=44), кадры ✅ (manifest=5817, actual=5817)
valid: пациенты ✅ (manifest=10, actual=10), кадры ✅ (manifest=1254, actual=1254)
test: пациенты ✅ (manifest=10, actual=10), кадры ✅ (manifest=1252, actual=1252)
